In [1]:
import importlib
import sys
import pickle

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

In [2]:
#load model
file_path_model = '../../../training_variational_dropout_v2/BPIC20_DD/BPIC20_DD_full_grad_norm_pro_conf_check_v2.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.1)

# Load the dataset
# file_path_data_set = '../../../../../encoded_data_conformance_v2/BPIC20_DD/bpic20_DD_all_5_val.pkl'
# helpdesk_test_dataset = torch.load(file_path_data_set, weights_only=False)

file_path_data_set = '../../../../../encoded_data_conformance_v2/BPIC20_DD/bpic20_DD_all_5_test.pkl'
helpdesk_test_dataset = torch.load(file_path_data_set, weights_only=False)

Dynamic data set categories:  ([('concept:name', 10, {'Declaration APPROVED': 1, 'Declaration FINAL_APPROVED': 2, 'Declaration FOR_APPROVAL': 3, 'Declaration REJECTED': 4, 'Declaration SAVED': 5, 'Declaration SUBMITTED': 6, 'EOS': 7, 'Payment Handled': 8, 'Request Payment': 9}), ('org:resource', 4, {'EOS': 1, 'STAFF MEMBER': 2, 'SYSTEM': 3}), ('org:role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Data set static categories:  ([('case:BudgetNumber', 2, {'budget 86566': 1}), ('case:DeclarationNumber', 5515, {'UNKNOWN': 1, 'declaration number 100001': 2, 'declaration number 100011': 3, 'declaration number 100016': 4, 'declaration number 100038': 5, 'declaration number 100070': 6, 'declaration number 100075': 7, 'declaration number 100080': 8, 'declaration number 100091': 9, 'declara

In [3]:
import evaluation_v2.probabilistic_evaluation
importlib.reload(evaluation_v2.probabilistic_evaluation)
from evaluation_v2.probabilistic_evaluation import ProbabilisticEvaluation

new_eval = ProbabilisticEvaluation(model=model, 
                                   dataset=helpdesk_test_dataset,
                                   concept_name='concept:name',
                                   num_processes=32,
                                   growing_num_values = [],
                                   # growing_num_values = ['case_elapsed_time'],
                                   # number of samples
                                   samples_per_case = 100,
                                   sample_argmax = False,
                                   use_variance_cat = True,
                                   use_variance_num = True,
                                   decoder_cat=['concept:name'],
                                   decoder_num=['event_elapsed_time']
                                   )

In [4]:
def save_chunk(results, i):
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

# output_dir = '../../../../../../../data/BPIC20/proact_conf_check_v2/eval_validation_set'
output_dir = '../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set'

save_every = 50

results = {}
#for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate(random_order=True)):
for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate_multi_processing(random_order=True)):
    # print(case_name, prefix_len)
    assert((case_name, prefix_len) not in results)
    results[(case_name, prefix_len)] = (prefix, suffix, mean_prediction, predicted_suffixes)
    # print(prefix_len, len(suffix))
    if (i + 1) % save_every == 0:
        save_chunk(results, i)
        results = {}

if len(results):
    save_chunk(results, i)

  0%|          | 0/1575 [00:00<?, ?it/s]

Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_050.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_100.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_150.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_200.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_250.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_300.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_350.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_400.pkl
Saved 50 results to ../../../../../../../data/BPIC20/proact_conf_check_v2/eval_test_set/results_part_450.pkl
Saved 50 results to